In [0]:
# Install required library
%pip install --quiet pyyaml

import yaml

# === USER VARIABLE: YAML file path ===
yaml_file_path = './source_files.yaml'  # Update this path as needed

# Load the YAML file
with open(yaml_file_path, 'r') as f:
    yaml_data = yaml.safe_load(f)

# Extract the list and category mapping
# source_files is loaded from YAML in Cell 1
# Example structure:
# source_files = [
#     {'file_name': ..., 'gdrive_link': ..., 'category': ...},
#     ...
# ]
source_files = yaml_data['source_files']
category_to_dir = yaml_data['category_to_dir']
print(f"Loaded {len(source_files)} files from YAML.")
print(f"Loaded {len(category_to_dir)} category-to-directory mappings from YAML.")

In [0]:
# Create all directories from the paths in category_to_dir using dbutils.fs.mkdirs
for category, path in category_to_dir.items():
    # Example path: /Volumes/jaccueille/bronze/source_files/logement/loyer/
    try:
        dbutils.fs.mkdirs(path)
        print(f"Ensured directory exists: {path}")
    except Exception as e:
        print(f"Error creating directory {path}: {e}")

# Méthodes d’Ingestion de Données Google Drive

Ce notebook fournit des méthodes pour ingérer des fichiers de données depuis Google Drive dans votre environnement Databricks. Il prend en charge le téléchargement de fichiers à partir de liens Google Drive publics et leur organisation dans des volumes Unity Catalog selon leur catégorie.

**Fonctionnalités principales :**
- Convertit les liens de partage Google Drive en liens de téléchargement direct.
- Télécharge les fichiers en parallèle pour plus d’efficacité.
- Organise les fichiers dans des répertoires selon leur catégorie (par exemple : logement, transport...).
- Utilise un fichier de configuration YAML pour gérer les métadonnées des fichiers et les répertoires cibles.

In [0]:
import requests
import re
import os

def gdrive_extract_file_id(link: str) -> str:
    """Extract file ID from Google Drive share link."""
    match = re.search(r'/d/([a-zA-Z0-9_-]+)', link)
    if not match:
        raise ValueError(f"Could not extract file ID from link: {link}")
    return match.group(1)


def download_gdrive_file_to_volume(file_id: str, destination_path: str):
    """
    Download a Google Drive file (small or large) by handling the virus scan warning page.
    """
    session = requests.Session()
    base_url = "https://drive.google.com/uc?export=download"

    response = session.get(base_url, params={'id': file_id}, stream=True)
    response.raise_for_status()

    # If we got HTML instead of binary -> parse hidden form
    if "text/html" in response.headers.get("Content-Type", ""):
        html = response.text

        # Find form action
        form_action = re.search(r'<form[^>]+action="([^"]+)"', html)
        if not form_action:
            raise RuntimeError("Could not find download form in Google Drive response")
        action_url = form_action.group(1)

        # Find all hidden input fields
        inputs = dict(re.findall(r'name="([^"]+)" value="([^"]*)"', html))

        # Retry actual download
        response = session.get(action_url, params=inputs, stream=True)
        response.raise_for_status()

    # Write file
    os.makedirs(os.path.dirname(destination_path), exist_ok=True)
    with open(destination_path, "wb") as f:
        for chunk in response.iter_content(32768):
            if chunk:
                f.write(chunk)


In [0]:
import concurrent.futures

MAX_PARALLEL_DOWNLOADS = 8

def download_file_entry(file_info):
    category = file_info.get("category")
    if not category or category not in category_to_dir:
        return f"Skipping {file_info['file_name']}: Invalid or missing category '{category}'."

    destination_dir = category_to_dir[category]
    destination_path = os.path.join(destination_dir, file_info["file_name"])

    try:
        file_id = gdrive_extract_file_id(file_info["gdrive_link"])
    except Exception as e:
        return f"Skipping {file_info['file_name']}: {e}"

    try:
        download_gdrive_file_to_volume(file_id=file_id, destination_path=destination_path)
        return f"Downloaded {file_info['file_name']} to {destination_path}"
    except Exception as e:
        return f"Failed to download {file_info['file_name']}: {e}"


results = []
with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_PARALLEL_DOWNLOADS) as executor:
    future_to_file = {executor.submit(download_file_entry, file_info): file_info for file_info in source_files}
    for future in concurrent.futures.as_completed(future_to_file):
        result = future.result()
        print(result)
        results.append(result)